In [9]:
from argparse import ArgumentParser, Namespace
import Spectral_Preprocessing as sp
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import json

In [41]:
parser = ArgumentParser()
parser.add_argument("--range-ir-1", dest= "wn_range_ir_1", default = (948,1860))
parser.add_argument("--range-ir-2", dest= "wn_range_ir_2", default = (2698,3002))

parser.add_argument("--range-min-ir", dest= "range_min_ir", type = int, default = 400)
parser.add_argument("--range-max-ir", dest= "range_max_ir", type = int, default = 4000)
parser.add_argument("--interpolate-interval-ir", dest = "interpolate_interval_ir", type = int, default = 1) 

parser.add_argument("--finger-print-exl", dest= "finger_print_exl", type = bool, default = False)
parser.add_argument("--trans-chip", dest= "trans_chip", type = bool, default = False)

parser.add_argument("--background-rule-ir", dest= "background_rule", type = str, default = 'mean')
parser.add_argument("--threshold-mean-intensity-ir", dest= "ths_mean_int_ir", type = float, default = 0)
parser.add_argument("--threshold-total-variation-intensity-ir", dest= "ths_tot_var_int_ir", type = float, default = 0)

parser.add_argument("--range-min-raman", dest= "range_min_raman", type = float, default = 400)
parser.add_argument("--range-max-raman", dest= "range_max_raman", type = float, default = 4000)
parser.add_argument("--interpolate-interval-raman", dest = "interpolate_interval_raman", type = float, default = 1) 

parser.add_argument("--rubberband", dest="rubberband", action="store_true", default=False,)
parser.add_argument("--minmax", dest="minmax", action="store_true", default=False,)
parser.add_argument("--no-interpolate", dest = "no_interpolate", action="store_true", default = False)
parser.add_argument("--nitrogen-calibration", dest="nitrogen_calibration", action="store_true", default=True,)
parser.add_argument("--si-peak", dest= "si_peak", type = float, default = 520)

parser.add_argument("--smooth", dest = "smooth", type = int, default = 0)
parser.add_argument("--baseline", dest = "baseline", type = int, default = 5)

###Sample Code #1: Specify IR and Raman interpolation intervals along with min & max values
args: Namespace = parser.parse_args([]) ###Do not alter this section.
###PLEASE TYPE IN YOUR ARGUMENTS HERE
args.wn_range_ir_1 = (980, 1800) #(675.0,1800)
args.wn_range_ir_2 = (2720, 2970) #(1800,4000)
args.range_min_ir = 675.0 #674.9918 ###enter an integer
args.range_max_ir = 4000.0 #3999.809 ###enter an integer [max value is not included]
args.interpolate_interval_ir = 2 #1.929 ###enter an integer
args.finger_print_exl = False
args.trans_chip = True
args.background_rule = 'total_var'
args.ths_mean_int_ir = 0.043
args.ths_tot_var_int_ir = 0.4543#0.59

args.range_min_raman = 355 #355 #.555542 ###enter an integer or float #530
args.range_max_raman = 3500 #3895.935059 ##enter an integer or float [max value is not included] 
args.interpolate_interval_raman = 2.0 #168021680216802 ###enter an integer or float

args.rubberband = 'specy'#'specy' ###True for rubberband, False for iModPoly
args.minmax = 'snv' ###False for SVT normalization, True for minmax
args.no_interpolate = False ###use sample code 2 if using True for this option.
args.nitrogen_calibration = True ##True for calibration, False for no calibration.
args.si_peak = 520 ###enter an integer or float

# active enviromental data

## MEC_04_014

In [42]:
calibrationLog = pd.read_excel('enviromental_data/mIRageRaman_CalibrationLog_RepeatsRemoved.xlsx')
#calibrationLog = calibrationLog.loc[["Cycled" not in x if type(x) is str else False for x in calibrationLog.iloc[:,-1]], :]
calibrationLog = calibrationLog.rename({'Notes (standard settings: high objective, 10 s integration, 3 averages, 0.5% power, no delay time)':'comments',
                                        'File name':'File_name'}, axis=1)
#duplicated_comments = ['20240913_AMA_RamanCalibration_Si2.ptir',
#                       '20240401_TZ_RamanCalibration_Si.ptir']

#calibrationLog = calibrationLog.query('File_name not in @duplicated_comments').reset_index(drop=True)


In [43]:
directory_name = 'enviromental_data/MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]


In [44]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak

array([520.53, 520.53, 520.53, 520.53, 520.53, 520.53, 520.53, 520.1 ,
       520.53, 520.53, 520.53, 520.53, 520.53, 520.1 ])

In [45]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Docum

In [46]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_1',
       'MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_2',
       'MEC_04_014_DC_Active_20240808_S4_GridSi_20241009_3', ...,
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_60',
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_61',
       'MEC_04_014_DC_Active_20240808_S1_GridSi_20250306_62'],
      dtype='<U55')

In [47]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())


In [48]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(175, 1018)

In [22]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(671, 522)

## MEC_04_023

In [49]:
directory_name = 'enviromental_data/MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]
files_list


['S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221.ptir',
 'S2/20250220/MEC_04_023_DC_Active_20241212_S2_GridSi_20250220.ptir',
 'S2/20250207/MEC_04_023_DC_Active_20241212_S2_GridSi_20250207.ptir',
 'S3/20250130/MEC_04_023_DC_Active_20241212_S3_GridSi_20250130.ptir',
 'S3/20250122/MEC_04_023_DC_Active_20241212_S3_GridSi_20250122.ptir',
 'S4/20241218/MEC_04_023_DC_Active_20241212_S4_GridSi_20241218.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250227.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250221.ptir',
 'S1/MEC_04_023_DC_Active_20241212_S1_GridSi_20250228.ptir',
 'S6/20250130/MEC_04_023_DC_Active_20241212_S6_GThinSi_20250130.ptir',
 'S6/20250131/MEC_04_023_DC_Active_20241212_S6_ThinSi_20250131.ptir']

In [50]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
#args.si_peak = [520]*len(files_list)
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak

array([520.1 , 520.1 , 520.1 , 520.1 , 520.1 , 520.53, 520.1 , 520.1 ,
       520.1 , 520.1 , 520.1 ])

In [51]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/U

In [52]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_1',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_2',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_3',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_4',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_5',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_6',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_7',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_8',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_9',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_10',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_11',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_12',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_13',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_14',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_15',
       'MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_16',
       'MEC_04_02

In [53]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())

In [54]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(51, 902)

In [28]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(698, 255)

## MEC_02_119

In [55]:
directory_name = 'enviromental_data/MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-4:]=="ptir"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-4:]=="ptir"]
files_list


['S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203.ptir',
 'S2/20241204/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241204.ptir',
 'S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522.ptir',
 'S3/20240502/MEC_02_119_DC_Sp_S3_GridSi_FOV1and2_20240502.ptir',
 'S3/20241127/MEC_02_119_DC_Sp_S3_GridSi_20241127.ptir',
 'S3/20240509/MEC_02_119_DC_Sp_S3_GridSi_FOV3_20240509.ptir',
 'S3/20241011/MEC_02_119_DC_Sp_S3_GridSi_20241011.ptir',
 'S3/20240522/MEC_02_119_DC_Sp_S3_GridSi_FOV4,5,6,7,8_20240522.ptir',
 'S4/MEC_02_119_DC_Active_20240418_S4_GridSi_20241030.ptir',
 'S4/MEC_02_119_DC_Sp_S4_GridSi_FOV1_20240424.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240305.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240228.ptir',
 'S1/MEC_02_119_DC_Sp_S1_GridSi_20240306.ptir',
 'S6/ThinSi/MEC_02_119_DC_Active_20240418_S6_ThinSi_20241023.ptir',
 'S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_20240502.ptir']

In [56]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
#args.si_peak = [520]*len(files_list)
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak

array([520.53, 520.53, 524.  , 517.13, 520.53, 517.13, 520.53, 524.  ,
       520.53, 517.9 , 520.53, 548.31, 520.53, 520.53, 517.13])

In [57]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: invalid value encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = a

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]


In [58]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U55')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name

array(['MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_1',
       'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_2',
       'MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_3', ...,
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133',
       'MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_134'], dtype='<U55')

In [59]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())


In [60]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(170, 1005)

In [34]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(768, 407)

# passive enviromental data

In [61]:
directory_name = 'enviromental_data/MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/' 
ir_export_filename =  "enviromental_data/no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "enviromental_data/no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'enviromental_data/MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'  #'MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'
folders = os.listdir(path)
files_list = []

for folder in folders:
    if folder[-4:] in ["xlsx", "tore"]:
        continue
    dirs = os.listdir(path+folder)
    for d in dirs:
        if d[-4:] in ["tore"]:
            continue
        files = os.listdir(path+folder+'/'+d)
        files_list = files_list+[folder+'/'+d+'/'+f for f in files if f[-5:] == '.ptir']

files_list

['08 August 2024/FHS_20240911/FHS_passive_August2024_B4_20240910.ptir',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_20240911.ptir',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_20240917.ptir',
 '08 August 2024/ICB_20240918/ICB_passive_August2024_E8_20240918.ptir',
 '08 August 2024/DC_20240910/DC_passive_August2024_J10_20240910.ptir',
 '07 July 2024/FHS_20240807/FHS_passive_July2024_D6_20240813.ptir',
 '07 July 2024/FHS_20240807/FHS_passive_July2024_D6_20240807.ptir',
 '07 July 2024/IAO_20240813/IAO_passive_July2024_F10_20240813.ptir',
 '07 July 2024/DC_20240806/DC_passive_July2024_I9_20240806.ptir',
 '07 July 2024/ICB_20240814/ICB_passive_July2024_C4_20240814.ptir',
 '11 November 2024/ICB_20241126/ICB_passive_November2024_C10_20241126.ptir',
 '11 November 2024/DC_20241113/DC_passive_November2024_C5_20241119.ptir',
 '11 November 2024/DC_20241113/DC_passive_November2024_C5_20241113.ptir',
 '11 November 2024/FHS_20241119/FHS_passive_November2024_I5_20241119.ptir',


In [62]:
files_df = pd.DataFrame({'file':files_list,
                         'Date':[np.datetime64(x[-13:-9]+'-'+x[-9:-7]+'-'+x[-7:-5]) for x in files_list]})

files_df = files_df.merge(calibrationLog.loc[:,['Date', 'Raman shift (cm-1)']], how='left')
#args.si_peak = [520]*len(files_list)
args.si_peak = np.array(files_df['Raman shift (cm-1)'])
args.si_peak[np.isnan(args.si_peak)] = 520.53
args.si_peak

array([520.53, 520.53, 520.53, 520.53, 520.53, 520.1 , 516.63, 520.1 ,
       520.1 , 520.1 , 520.53, 520.53, 520.53, 520.53, 520.53, 514.44,
       520.53, 524.  , 520.53, 520.53, 520.53, 520.53, 520.53, 520.53,
       520.53, 512.65, 516.08, 524.  , 520.53, 520.53, 520.53, 520.53,
       520.53, 520.53, 520.53, 520.53, 520.53, 520.53, 512.65, 524.  ,
       520.53, 520.1 , 520.53, 520.53, 520.53, 520.1 ])

In [63]:
(raman_final, 
 ir_final, 
 label_list, 
 XX_ir, 
 XX_raman, 
 ids, 
 coords, 
 _, 
 rng, 
 ir_bg_label, 
 saturated_wn_list) = sp.ptir_to_interpolated_processed_csv(args, 
                                                            directory_name,
                                                            ir_export_filename, 
                                                            raman_export_filename, 
                                                            file_list = files_list, 
                                                            transpose = True)

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: invalid value encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/User

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: invalid value encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:396: RuntimeWarning: invalid value encountered in scalar divide
  data4 = [(element - np.mean(data2)) / np.std(data2) for element in data2]
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: invalid value encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/User

/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: divide by zero encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:229: RuntimeWarning: invalid value encountered in scalar divide
  criteria = abs((dev_curr - devPrev) / dev_curr) <= 0.05
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/Spectral_Preprocessing.py:187: RuntimeWarning: invalid value encountered in divide
  return (x - x.mean()) / x.std()
/Users/eochoa/Documents/Research_Ambu

In [64]:
ids_file_name_ = np.array([x.split('/')[-1][:-5] for x in ids])
ids_file_name = np.array([x.split('/')[-1][:-5] for x in ids],dtype='<U50')
ids_file_name_list = np.unique(ids_file_name)
idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1
ids_file_name.shape

(8186,)

In [24]:
#print(np.max(np.concatenate([XX_raman[x] for x in saturated_wn_list if x.shape[0]<15])))
#plt.hist(np.concatenate([XX_raman[x] for x in saturated_wn_list if x.shape[0]<15], 0))

In [25]:
#plt.plot(pd.DataFrame({'wn':np.concatenate([XX_raman[x] for x in saturated_wn_list if x.shape[0]<15], 0),
# 'n':1}).groupby('wn').sum())

In [26]:
pd.DataFrame({'file_name':[x.split('/')[-1][:-5] for x in ids],
              'n':1}).groupby('file_name').sum()

,n
file_name,
DC_passive_AprMay2024_G7_20240529,219
DC_passive_August2024_J10_20240910,203
DC_passive_December2024_H5_20241217,202
DC_passive_July2024_I9_20240806,223
DC_passive_June2024_I7_20240717,217
DC_passive_MarApr2024_C9_20240612,204
DC_passive_May2024_D2_20240702,99
DC_passive_May2024_D2_20240703,104
DC_passive_November2024_C5_20241113,180


In [27]:
#trunc_saturated_wn_list = [x[XX_raman[x]>600] for x in saturated_wn_list]
#trunc_saturated_wn_list

In [28]:
#plt.hist([x.shape[0] for x in trunc_saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in trunc_saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [29]:
#plt.hist([x.shape[0] for x in saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [30]:
#plt.hist([x.shape[0] for x in saturated_wn_list])
#pd.DataFrame({'n':[x.shape[0] for x in saturated_wn_list],
# 'x':1}).groupby('n').sum()#.iloc[:16].sum()#/ len(saturated_wn_list)

In [65]:
nas_list = np.unique(np.where(np.isnan(raman_final))[0])
non_nas_idx = [x not in nas_list for x in range(raman_final.shape[0])]

saturated_wn_dict = {id_:str(list(x))[1:-1] for x, id_ in zip(saturated_wn_list,ids_file_name) if x.shape[0]>0}
ids_sat = list(saturated_wn_dict.keys())


In [66]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(334, 7852)

In [40]:
np.sum([x.shape[0]==0 for x in saturated_wn_list]), np.sum([x.shape[0]>0 for x in saturated_wn_list])

(6401, 1785)